# PyEncode — Paper Experiments
Reproduces all examples, figures, and gate count table from the paper.
Uses the primary API: `encode()` with `SPARSE`, `STEP`, `SQUARE`, `FOURIER`.

## 1. Imports and Configuration

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import warnings
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

from qiskit import transpile, QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit.circuit.library import StatePreparation

from pyencode import (
    encode,
    SPARSE, STEP, SQUARE, FOURIER,
    EncodingInfo, VectorType,
)

N = 64
k = np.arange(N)

# Transpilation settings (must match paper footnote)
BASIS = ["cx", "u", "x", "h", "ry", "rz", "rx", "p"]
OPTIMIZATION_LEVEL = 3


## 2. Helper Functions

In [ ]:
def plot_vector(f, title, smooth=False):
    """Plot a vector with bar or smooth style."""
    fig, ax = plt.subplots(figsize=(6, 2.5))
    x = np.arange(len(f))
    if smooth:
        ax.plot(x, f, "steelblue", lw=1.5)
        ax.fill_between(x, 0, f, alpha=0.25)
    else:
        ax.bar(x, f, color="steelblue", width=0.7)
    ax.set_title(title)
    ax.set_xlabel("index $i$")
    plt.tight_layout(); plt.show()

def verify_circuit(circuit, info, label=""):
    """Simulate and verify circuit matches expected statevector."""
    sv = np.abs(np.array(Statevector(circuit)))
    print(f"{label}: gate_count={info.gate_count}, complexity={info.complexity}")
    return sv

def qiskit_gate_count(f):
    """Qiskit StatePreparation gate count on the same vector."""
    norm = np.linalg.norm(f)
    if norm < 1e-14: return 0
    sv = (f / norm).astype(complex)
    qc = QuantumCircuit(int(round(np.log2(len(f)))))
    qc.append(StatePreparation(sv), range(qc.num_qubits))
    t = transpile(qc.decompose(reps=3), basis_gates=BASIS,
                  optimization_level=OPTIMIZATION_LEVEL)
    return sum(t.count_ops().values())


---
## 3. All Vector Types
### 3.1 SPARSE — Gleinig-Hoefler $\mathcal{O}(sm)$

In [ ]:
# s=1: motivating example from paper (index 19, N=64)
circuit, info = encode(SPARSE([(19, 1.0)]), N=N)
print(info)

f = np.zeros(N); f[19] = 1.0
plot_vector(f, r"SPARSE s=1: $f_{19}=1$, $N=64$")
verify_circuit(circuit, info, "SPARSE s=1")
circuit.draw("mpl")


In [ ]:
# s=2: two point masses
circuit, info = encode(SPARSE([(10, 3.0), (50, 4.0)]), N=N)
print(info)

f = np.zeros(N); f[10] = 3.0; f[50] = 4.0
plot_vector(f, r"SPARSE s=2: $f_{10}=3,\;f_{50}=4$")
verify_circuit(circuit, info, "SPARSE s=2")
circuit.draw("mpl")


### 3.2 STEP — Shukla & Vedula $\mathcal{O}(m)$

In [ ]:
circuit, info = encode(STEP(k_s=4, c=1.0), N=N)
print(info)

f = np.zeros(N); f[:4] = 1.0
plot_vector(f, r"STEP: $f_{[:4]}=1$")
verify_circuit(circuit, info, "STEP k_s=4")
circuit.draw("mpl")


In [ ]:
# Non-power-of-2 boundary
circuit, info = encode(STEP(k_s=37, c=1.0), N=N)
print(f"STEP k_s=37: gate count = {info.gate_count}")
f = np.zeros(N); f[:37] = 1.0
plot_vector(f, r"STEP: $f_{[:37]}=1$")


### 3.3 SQUARE — this work $\mathcal{O}(m)$

In [ ]:
circuit, info = encode(SQUARE(k1=8, k2=16, c=1.0), N=N)
print(info)

f = np.zeros(N); f[8:16] = 1.0
plot_vector(f, r"SQUARE: $f_{[8:16)}=1$")
verify_circuit(circuit, info, "SQUARE [8,16)")
circuit.draw("mpl")


In [ ]:
# Non-aligned square (general case)
circuit, info = encode(SQUARE(k1=10, k2=50, c=1.0), N=N)
print(f"SQUARE [10,50): gate count = {info.gate_count}")
f = np.zeros(N); f[10:50] = 1.0
plot_vector(f, r"SQUARE: $f_{[10:50)}=1$")


### 3.4 FOURIER — inverse QFT $\mathcal{O}(m^2)$

In [ ]:
import math
# T=1: sine (phi=0)
circuit, info = encode(FOURIER(modes=[(1, 1.0, 0)]), N=N)
print(info)

f = np.sin(2 * np.pi * k / N)
plot_vector(f, r"FOURIER T=1: $\sin(2\pi i/N)$", smooth=True)
verify_circuit(circuit, info, "FOURIER T=1 sine")
circuit.draw("mpl")


In [ ]:
# T=1: cosine (phi=pi/2)
circuit, info = encode(FOURIER(modes=[(1, 1.0, math.pi/2)]), N=N)
print(f"FOURIER cosine: gate count = {info.gate_count}")

f = np.cos(2 * np.pi * k / N)
plot_vector(f, r"FOURIER T=1: $\cos(2\pi i/N)$", smooth=True)


In [ ]:
# T=1: n=3, phi=pi/4
circuit, info = encode(FOURIER(modes=[(3, 2.0, math.pi/4)]), N=N)
print(info)

f = 2.0 * np.sin(2 * np.pi * 3 * k / N + math.pi/4)
plot_vector(f, r"FOURIER T=1: $2\sin(6\pi i/N + \pi/4)$", smooth=True)


In [ ]:
# T=2: multi-mode
circuit, info = encode(FOURIER(modes=[(1, 2.0, 0), (3, 1.0, 0)]), N=N)
print(info)

f = 2.0*np.sin(2*np.pi*k/N) + np.sin(2*np.pi*3*k/N)
plot_vector(f, r"FOURIER T=2: $2\sin(2\pi i/N) + \sin(6\pi i/N)$", smooth=True)
verify_circuit(circuit, info, "FOURIER T=2")


### 3.5 Qiskit Fallback — unrecognized pattern

In [ ]:
# Damped sine: not in the pattern library -> user falls back explicitly
x = np.linspace(0, 1, N)
f_damped = np.sin(3 * 2*np.pi * x) * np.exp(-x)
f_damped /= np.linalg.norm(f_damped)

qc = QuantumCircuit(int(np.log2(N)))
qc.append(StatePreparation(f_damped.astype(complex)), range(qc.num_qubits))
qc = qc.decompose(reps=3)
t = transpile(qc, basis_gates=BASIS, optimization_level=OPTIMIZATION_LEVEL)
print(f"Qiskit fallback: {sum(t.count_ops().values())} gates (O(2^m))")
plot_vector(f_damped, r"Damped sine: $\sin(6\pi x)e^{-x}$", smooth=True)


---
## 4. Gate Count Comparison (Table in Paper)

In [ ]:
def pyencode_counts(vobj):
    c, info = encode(vobj, N=N)
    raw = info.gate_count
    t = transpile(c, basis_gates=BASIS, optimization_level=OPTIMIZATION_LEVEL)
    transp = sum(t.count_ops().values())
    return raw, transp, info.complexity

cases = [
    ("SPARSE s=1 (k=20)",    SPARSE([(20, 1.0)]),              np.eye(N)[20]),
    ("STEP (k_s=4)",         STEP(k_s=4, c=1.0),              np.r_[np.ones(4),  np.zeros(N-4)]),
    ("SQUARE ([8,16))",      SQUARE(k1=8, k2=16, c=1.0),      np.r_[np.zeros(8), np.ones(8), np.zeros(N-16)]),
    ("FOURIER T=1 n=1",      FOURIER(modes=[(1, 1.0, 0)]),     np.sin(2*np.pi*k/N)),
    ("FOURIER T=1 n=3 phi",  FOURIER(modes=[(3, 1.0, np.pi/4)]), np.sin(2*np.pi*3*k/N+np.pi/4)),
    ("SPARSE s=2",           SPARSE([(10,3.0),(50,4.0)]),      None),
]

print(f"  {'Pattern':<26} {'Raw':>6} {'Transp':>8} {'Qiskit':>8}  Complexity")
print("-" * 65)
for desc, vobj, f_ref in cases:
    raw, transp, compl = pyencode_counts(vobj)
    if f_ref is not None:
        qk = qiskit_gate_count(f_ref)
    else:
        f_ref2 = np.zeros(N); f_ref2[10]=3.0; f_ref2[50]=4.0
        qk = qiskit_gate_count(f_ref2)
    print(f"  {desc:<26} {raw:>6} {transp:>8} {qk:>8}  {compl}")


---
## 5. Application Examples
### 5.1 Quantum Chemistry: Fermi-Hubbard PREP

In [ ]:
L = 8; t_hop = 1.0; U_int = 4.0
N_pauli = 4 * L  # = 32

circuit, info = encode([
    SQUARE(k1=0,   k2=2*L, c=t_hop),
    SQUARE(k1=2*L, k2=3*L, c=U_int),
], N=N_pauli)
print(info)

f = np.zeros(N_pauli); f[0:2*L] = t_hop; f[2*L:3*L] = U_int
plot_vector(f, f"Fermi-Hubbard PREP: t={t_hop}, U={U_int}, L={L}")


### 5.2 Computational Mechanics: 2D Poisson

In [ ]:
N_grid = 32
k_grid = np.arange(N_grid)

circ_x, info_x = encode(FOURIER(modes=[(2, 1.0, 0)]), N=N_grid)
circ_y, info_y = encode(FOURIER(modes=[(3, 1.0, 0)]), N=N_grid)
circuit = circ_x.tensor(circ_y)

print(f"x-register: {info_x.gate_count} gates")
print(f"y-register: {info_y.gate_count} gates")
print(f"Total: {info_x.gate_count + info_y.gate_count} gates for {N_grid**2} amplitudes")

u = np.sin(2*np.pi*2*k_grid/N_grid)
v = np.sin(2*np.pi*3*k_grid/N_grid)
f_2d = np.outer(u, v)
plt.figure(figsize=(5,4))
plt.imshow(f_2d, cmap="RdBu_r", origin="lower")
plt.colorbar(); plt.title(r"2D Poisson source $\sin(4\pi x)\sin(6\pi y)$")
plt.show()


### 5.3 Quantum Finance: Price Distribution

In [ ]:
indices = [8, 16, 24, 32, 40, 48, 56]
weights = [0.05, 0.15, 0.25, 0.30, 0.15, 0.07, 0.03]

circuit, info = encode(
    SPARSE(list(zip(indices, weights))),
    N=N,
)
print(info)

f = np.zeros(N)
for k_i, w_i in zip(indices, weights): f[k_i] = w_i
plot_vector(f, "Price distribution (SPARSE, 7 bins)")

qk = qiskit_gate_count(f)
print(f"PyEncode: {info.gate_count} gates  |  Qiskit: {qk} gates")


---
## 6. Circuit Code Output

In [ ]:
_, info = encode(FOURIER(modes=[(1, 1.0, 0)]), N=16)
print(info.circuit_code)
